<a href="https://colab.research.google.com/github/RubaEgbaria/Esch-Crowd-Monitoring/blob/master/vid_rectangle_cleaner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## This code removes all green detection rectangles from a crowd monitoring video footage, leaving only the clean original scene.


### The morphology-based cleaning method works in three stages:

1. Detection: Identify green bounding box pixels using HSV color space thresholding.

2. Preprocessing: Apply morphological operations (dilation and closure) to expand detected regions and fill small gaps, ensuring complete rectangle coverage.

3. Reconstruction: Use image inpainting (Telea algorithm) to seamlessly reconstruct the underlying background by interpolating from surrounding pixel neighborhoods.

This approach preserves scene detail while completely removing detection overlays, producing clean footage.


In [1]:
import cv2
import numpy as np

class VideoRectangleCleaner:
    def __init__(self, input_video_path, output_video_path):
        """
        Initialize video cleaner

        Args:
            input_video_path: Path to original video with green rectangles
            output_video_path: Path to save cleaned video
        """
        self.input_path = input_video_path
        self.output_path = output_video_path

        # Open input video
        self.cap = cv2.VideoCapture(input_video_path)
        self.fps = self.cap.get(cv2.CAP_PROP_FPS)
        self.width = int(self.cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        self.height = int(self.cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        self.total_frames = int(self.cap.get(cv2.CAP_PROP_FRAME_COUNT))

        # Define codec and create VideoWriter
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        self.out = cv2.VideoWriter(output_video_path, fourcc, self.fps,
                                   (self.width, self.height))

    def create_green_mask(self, frame, lower_green=None, upper_green=None):
        """
        Create mask for green rectangles

        Args:
            frame: Current video frame
            lower_green: Lower HSV bounds for green
            upper_green: Upper HSV bounds for green

        Returns:
            Binary mask of green areas
        """
        if lower_green is None:
            lower_green = np.array([35, 40, 40])
        if upper_green is None:
            upper_green = np.array([85, 255, 255])

        hsv = cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)
        mask = cv2.inRange(hsv, lower_green, upper_green)

        return mask

    def clean_frame_inpaint(self, frame, mask, method='telea'):
        """
        Remove rectangles using inpainting

        Args:
            frame: Original frame
            mask: Binary mask of areas to inpaint
            method: 'telea' (fast) or 'ns' (slow but better quality)

        Returns:
            Cleaned frame
        """
        if method == 'telea':
            cleaned = cv2.inpaint(frame, mask, 3, cv2.INPAINT_TELEA)
        else:  # 'ns'
            cleaned = cv2.inpaint(frame, mask, 3, cv2.INPAINT_NS)

        return cleaned

    def clean_frame_morphology(self, frame, mask):
        """
        Alternative method: Remove green pixels and fill with neighborhood

        Args:
            frame: Original frame
            mask: Binary mask of green areas

        Returns:
            Cleaned frame
        """
        cleaned = frame.copy()

        # Dilate mask to ensure complete removal
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))
        dilated_mask = cv2.dilate(mask, kernel, iterations=2)

        # Apply morphological closing to fill small holes
        kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
        dilated_mask = cv2.morphologyEx(dilated_mask, cv2.MORPH_CLOSE, kernel)

        # Use inpainting on dilated mask for better results
        cleaned = cv2.inpaint(cleaned, dilated_mask.astype(np.uint8), 5, cv2.INPAINT_TELEA)

        return cleaned

    def process_video(self, method='morphology', show_progress=True):
        """
        Process entire video and remove rectangles

        Args:
            method: 'inpaint' or 'morphology' (morphology usually gives better results)
            show_progress: Print progress updates
        """
        frame_count = 0

        print(f"Processing video: {self.input_path}")
        print(f"Total frames: {self.total_frames}")
        print(f"Resolution: {self.width}x{self.height}")
        print(f"FPS: {self.fps}")
        print(f"Cleaning method: {method}")
        print("-" * 60)

        while True:
            ret, frame = self.cap.read()

            if not ret:
                break

            # Create mask for green rectangles
            mask = self.create_green_mask(frame)

            # Clean frame based on method
            if method == 'morphology':
                cleaned_frame = self.clean_frame_morphology(frame, mask)
            else:  # inpaint
                cleaned_frame = self.clean_frame_inpaint(frame, mask, method='telea')

            # Write cleaned frame to output video
            self.out.write(cleaned_frame)

            frame_count += 1

            if show_progress and frame_count % 50 == 0:
                progress = (frame_count / self.total_frames) * 100
                print(f"Progress: {frame_count}/{self.total_frames} frames ({progress:.1f}%)")

        # Release everything
        self.cap.release()
        self.out.release()

        print("-" * 60)
        print(f"✅ Cleaned video saved to: {self.output_path}")
        print(f"Total frames processed: {frame_count}")

    def process_video_with_preview(self, method='morphology', save_preview_frames=5):
        """
        Process video and save before/after comparison frames

        Args:
            method: Cleaning method
            save_preview_frames: Number of frames to save as preview
        """
        frame_count = 0
        saved_previews = 0

        print(f"Processing video with preview frames...")
        print("-" * 60)

        while True:
            ret, frame = self.cap.read()

            if not ret:
                break

            # Create mask
            mask = self.create_green_mask(frame)

            # Clean frame
            if method == 'morphology':
                cleaned_frame = self.clean_frame_morphology(frame, mask)
            else:
                cleaned_frame = self.clean_frame_inpaint(frame, mask)

            # Write to output
            self.out.write(cleaned_frame)

            # Save preview comparison every N frames
            if frame_count % (self.total_frames // save_preview_frames) == 0 and saved_previews < save_preview_frames:
                # Create side-by-side comparison
                comparison = np.hstack([frame, cleaned_frame])
                preview_path = f"preview_frame_{frame_count}.png"
                cv2.imwrite(preview_path, comparison)
                print(f"Preview saved: {preview_path} (Original | Cleaned)")
                saved_previews += 1

            frame_count += 1

            if frame_count % 50 == 0:
                progress = (frame_count / self.total_frames) * 100
                print(f"Progress: {progress:.1f}%")

        self.cap.release()
        self.out.release()

        print("-" * 60)
        print(f"✅ Cleaned video saved: {self.output_path}")
        print(f"✅ Preview frames saved for comparison")

In [5]:
cleaner = VideoRectangleCleaner('WhatsApp Video 2026-09-10 at 13.56.03.mp4', 'raw-vid.mp4')
cleaner.process_video(method='morphology')  # Removes all green rectangles

Processing video: WhatsApp Video 2026-09-10 at 13.56.03.mp4
Total frames: 632
Resolution: 1024x576
FPS: 11.566970686026295
Cleaning method: morphology
------------------------------------------------------------
Progress: 50/632 frames (7.9%)
Progress: 100/632 frames (15.8%)
Progress: 150/632 frames (23.7%)
Progress: 200/632 frames (31.6%)
Progress: 250/632 frames (39.6%)
Progress: 300/632 frames (47.5%)
Progress: 350/632 frames (55.4%)
Progress: 400/632 frames (63.3%)
Progress: 450/632 frames (71.2%)
Progress: 500/632 frames (79.1%)
Progress: 550/632 frames (87.0%)
Progress: 600/632 frames (94.9%)
------------------------------------------------------------
✅ Cleaned video saved to: raw-vid.mp4
Total frames processed: 632
